# EVANTHEA — 01 Dataset Audit & Protein Intelligence 🧬
**Phase 1 Deliverable**: Biological data provenance, coordinate reconciliation, leakage-resistant split verification, and ESM-2 biological representation inspection.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import pandas as pd
import numpy as np
import json

from evanthea.data.schemas import ProteinRecord
from evanthea.data.ingestion import ProteinGymIngestor
from evanthea.data.splits import create_random_split, create_position_held_out_split, create_mutation_order_split
from evanthea.protein.sequence import validate_amino_acid_string

## 1. Load Canonical GFP Reference Sequence (Sarkisyan et al. 2016)

In [ ]:
GFP_SARKISYAN_WT = (
    "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQ"
    "HDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNG"
    "IKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"
)
wt_seq = validate_amino_acid_string(GFP_SARKISYAN_WT)
protein = ProteinRecord(
    id="GFP_AEQVI_Sarkisyan_2016",
    name="Green Fluorescent Protein",
    wildtype_sequence=wt_seq,
    length=len(wt_seq),
    uniprot_accession="P42212"
)
print(f"Target: {protein.name} ({protein.id})")
print(f"Sequence Length: {protein.length} residues")

## 2. Ingest and Validate Variants

In [ ]:
sample_csv = Path("../tests/fixtures/sample_gfp_dms.csv")
ingestor = ProteinGymIngestor(protein=protein)
variants, summary = ingestor.ingest_csv(sample_csv)

print(f"Total variants: {summary.total_records}")
print(f"Valid variants: {summary.valid_records} ({summary.validity_rate*100:.1f}%)")
print(f"Singles: {summary.single_mutants}, Multi: {summary.multi_mutants}")

## 3. Verify Leakage Resistance Across Splits

In [ ]:
pos_split = create_position_held_out_split(protein.id, variants, sequence_length=protein.length, seed=42)
order_split = create_mutation_order_split(protein.id, variants, seed=42)

print("Position-held-out validation:", pos_split.leakage_verification)
print("Mutation-order validation:", order_split.leakage_verification)